# Setup & Imports

In [1]:
import pandas as pd
from pathlib import Path

# Reusable EDA Helper Functions

Functions for common data-quality checks across all datasets.

In [2]:
def column_summary(df):
    """

    Generate a column-level overview of a DataFrame

    Checks data types, missing values, repeated values, and unique values for each column.

    """

    summary = pd.DataFrame({
        'data_type': df.dtypes, 
        'missing_values': df.isna().sum(),
        'repeated_values': df.apply(lambda col: col.duplicated().sum()), 
        'unique_values': df.nunique()
    })

    return summary

In [3]:
def check_numeric_values(df, columns):

    """
    Generate summary statistics and data-quality checks for selected numeric columns.

    """

    numeric_data = df[columns]

    results = pd.DataFrame({
        'missing': numeric_data.isna().sum(),
        'negative': numeric_data.lt(0).sum(),
        'zero': numeric_data.eq(0).sum(), 
        'min': numeric_data.min(),
        'maximum': numeric_data.max(),
        'mean': numeric_data.mean().round(2),
        'median': numeric_data.median()
    })

    return results

In [4]:
def check_outliers(df, columns):

    """
    Detect potential outliers using the IQR method for one or more numeric columns.

    """

    numeric_data = df[columns]

    q1 = numeric_data.quantile(0.25)
    q3 = numeric_data.quantile(0.75)

    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (numeric_data.lt(lower_bound)) | 
        (numeric_data.gt(upper_bound))
    ).sum()

    results = pd.DataFrame({
        'lower_bound': lower_bound, 
        'upper_bound': upper_bound, 
        'outlier_count': outlier_count
    })

    return results

# Dataset & Overview

In [5]:
project_dir = Path.cwd()

if not (project_dir / "data").is_dir():
    project_dir = project_dir.parent

bronze_dir = project_dir / "data" / "bronze"

csv_files = list(bronze_dir.rglob("*.csv"))

for file in csv_files:
    print(file.relative_to(bronze_dir))

ecommerce_Brazil_dataset\olist_customers_dataset.csv
ecommerce_Brazil_dataset\olist_geolocation_dataset.csv
ecommerce_Brazil_dataset\olist_orders_dataset.csv
ecommerce_Brazil_dataset\olist_order_items_dataset.csv
ecommerce_Brazil_dataset\olist_order_payments_dataset.csv
ecommerce_Brazil_dataset\olist_order_reviews_dataset.csv
ecommerce_Brazil_dataset\olist_products_dataset.csv
ecommerce_Brazil_dataset\olist_sellers_dataset.csv
ecommerce_Brazil_dataset\product_category_name_translation.csv


In [6]:
for file in csv_files:
    df = pd.read_csv(file)

    print(f"File: {file.name}")
    print(f"Shape: {df.shape}")
    print(f"Duplicates: {df.duplicated().sum()}")
    print(f"Missing values: {df.isna().sum().sum()}")
    print("-" * 50)

File: olist_customers_dataset.csv
Shape: (99441, 5)
Duplicates: 0
Missing values: 0
--------------------------------------------------
File: olist_geolocation_dataset.csv
Shape: (1000163, 5)
Duplicates: 261831
Missing values: 0
--------------------------------------------------
File: olist_orders_dataset.csv
Shape: (99441, 8)
Duplicates: 0
Missing values: 4908
--------------------------------------------------
File: olist_order_items_dataset.csv
Shape: (112650, 7)
Duplicates: 0
Missing values: 0
--------------------------------------------------
File: olist_order_payments_dataset.csv
Shape: (103886, 5)
Duplicates: 0
Missing values: 0
--------------------------------------------------
File: olist_order_reviews_dataset.csv
Shape: (99224, 7)
Duplicates: 0
Missing values: 145903
--------------------------------------------------
File: olist_products_dataset.csv
Shape: (32951, 9)
Duplicates: 0
Missing values: 2448
--------------------------------------------------
File: olist_sellers_datase

In [7]:
# Create a dictionary to access CSV files by name instead of list index
csv_dict = {file.stem: file for file in csv_files}

# 1. Customer Dataset

Explore customer identifiers, data completeness, duplicate records and geographic information

In [8]:
df_customers = pd.read_csv(
    csv_dict['olist_customers_dataset']
)

print('Shape:', df_customers.shape)

display(df_customers.head())

Shape: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
display(column_summary(df_customers))

print('Fully duplicated rows:', 
    df_customers.duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
customer_id,str,0,0,99441
customer_unique_id,str,0,3345,96096
customer_zip_code_prefix,int64,0,84447,14994
customer_city,str,0,95322,4119
customer_state,str,0,99414,27


Fully duplicated rows: 0


### Customer Identifier Investigation

Investigate repeated customer_unique_id values to understand the relationship between customer_id and customer_unique_id

In [10]:
# Check uniqueness of customer_id
print(
    "Duplicated customer_id:",
    df_customers["customer_id"].duplicated().sum()
)

# Check repeated occurrences of customer_unique_id
print(
    "Repeated customer_unique_id occurrences:",
    df_customers["customer_unique_id"].duplicated().sum()
)

# Count distinct customers appearing more than once
customer_frequency = (
    df_customers.groupby("customer_unique_id").size()
)

print(
    "Customers appearing more than once:",
    customer_frequency.gt(1).sum()
)

# Display top 10 customers by frequency
display(
    customer_frequency
    .sort_values(ascending=False)
    .head(10)
)

Duplicated customer_id: 0
Repeated customer_unique_id occurrences: 3345
Customers appearing more than once: 2997


customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64

2,997 distinct customer_unique_id values appear more than once. This is consistent with the Olist data model, where a customer can have multiple order-associated customer_id records. The maximum observed frequency is 17.

### Geographic Data Validation

Investigate customer distribution across Brazilian states and validate ZIP-code prefix lengths.

In [11]:
# Customer distribution by state
customer_states = (
    df_customers["customer_state"]
    .value_counts()
    .reset_index()
)

display(customer_states)

# ZIP-code prefix length distribution
zip_prefix_lengths = (
    df_customers["customer_zip_code_prefix"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

display(zip_prefix_lengths)

,customer_state,count
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


customer_zip_code_prefix
4    23995
5    75446
Name: count, dtype: int64

# 2. Orders Dataset

In [12]:
df_orders = pd.read_csv(
    csv_dict['olist_orders_dataset']
)

print('Shape:', df_orders.shape)
display(df_orders.head())

Shape: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [13]:
display(column_summary(df_orders))

print(
    "Fully duplicated rows:",
    df_orders.duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
order_id,str,0,0,99441
customer_id,str,0,0,99441
order_status,str,0,99433,8
order_purchase_timestamp,str,0,566,98875
order_approved_at,str,160,8707,90733
order_delivered_carrier_date,str,1783,18422,81018
order_delivered_customer_date,str,2965,3776,95664
order_estimated_delivery_date,str,0,98982,459


Fully duplicated rows: 0


In [14]:
print(
    "Duplicated order IDs:",
    df_orders["order_id"].duplicated().sum()
)

print(
    "Duplicated customer IDs:",
    df_orders["customer_id"].duplicated().sum()
)

Duplicated order IDs: 0
Duplicated customer IDs: 0


### Checking chronological order of timestamps

#### Converting from string to timestamp

In [15]:
# Since the column data are of type string, we are converting them to timestamp

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at', 
    'order_delivered_customer_date', 
    'order_delivered_carrier_date',
    'order_estimated_delivery_date'
]

df_orders[date_columns] = df_orders[date_columns].apply(pd.to_datetime)
df_orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_customer_date    datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

#### Checking Invalid Approval date

In [16]:
invalid_approval = df_orders[
    df_orders['order_approved_at'] <
    df_orders['order_purchase_timestamp']
]

print("Invalid approval dates:", len(invalid_approval))

display(invalid_approval[
    ["order_id", "order_purchase_timestamp", "order_approved_at"]
].head(10))

Invalid approval dates: 0


,order_id,order_purchase_timestamp,order_approved_at


#### Carrier Date earlier than approval Date

In [17]:
invalid_carrier = df_orders[
    df_orders['order_delivered_carrier_date'] <
    df_orders['order_approved_at']
]

invalid_customer_delivery_date = df_orders[
    df_orders['order_delivered_customer_date']<
    df_orders['order_delivered_carrier_date']
]

print("Invalid delivery at carriers date:", len(invalid_carrier))
print('Invalid customer delivery date:', len(invalid_customer_delivery_date))
len(invalid_customer_delivery_date)

Invalid delivery at carriers date: 1359
Invalid customer delivery date: 23


23

In [18]:
display(invalid_carrier[
    ["order_id", "order_delivered_carrier_date", "order_approved_at"]
].head(10))

,order_id,order_delivered_carrier_date,order_approved_at
15,dcb36b511fcac050b97cd5c05de84dc3,2018-06-11 14:54:00,2018-06-12 23:31:02
64,688052146432ef8253587b930b01a06d,2018-04-23 19:19:14,2018-04-24 18:25:22
199,58d4c4747ee059eeeb865b349b41f53a,2018-07-24 12:57:00,2018-07-26 23:31:53
210,412fccb2b44a99b36714bca3fef8ad7b,2018-07-23 12:24:00,2018-07-23 12:31:53
415,56a4ac10a4a8f2ba7693523bb439eede,2018-07-24 14:03:00,2018-07-27 23:31:09
481,32e4fa9bb468884309b58b9348de70c3,2018-07-05 14:50:00,2018-07-05 16:33:06
483,4df92d82d79c3b52c7138679fa9b07fc,2018-07-26 14:46:00,2018-07-29 23:30:52
585,16e38caa92e342c7780f68832f832d4d,2018-05-07 15:09:00,2018-05-07 16:52:39
615,b9afddbdcfadc9a87b41a83271c3e888,2018-08-16 13:27:00,2018-08-16 14:05:13
817,6051e6d3da9a50b7325cbe9c81025062,2018-07-04 12:14:00,2018-07-05 16:31:26


In [19]:
display(invalid_customer_delivery_date[
    ['order_id', 'order_delivered_customer_date', 'order_delivered_carrier_date']
].head(10))

,order_id,order_delivered_customer_date,order_delivered_carrier_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-25 19:32:56,2017-07-28 16:57:58
9553,383aa8b2724fe452d9ccd9934a8c628b,2017-07-06 14:27:51,2017-07-07 17:22:41
13487,cb1134f9010d242e9515ad1c78ec0c39,2017-07-19 14:13:28,2017-07-20 19:22:02
14474,dceb62e8fa94b46006c9554fed743df0,2017-07-26 18:09:10,2017-08-01 18:23:30
19268,5f9d46795c3126674e52becb3a1a517f,2017-07-20 18:52:41,2017-07-20 23:03:42
21338,8c78d01de3a9009e23d6877a7cc9be20,2016-10-25 17:51:46,2016-10-26 11:41:53
22520,b27af682321527a6349f1761eb3f360c,2017-06-26 15:45:35,2017-06-27 14:51:54
25393,1cc3ae63caffff2d6c3ee3e78e074acf,2017-08-10 18:05:38,2017-08-10 18:28:56
25646,e37f11cae9985ca58f0b56f268720537,2017-07-31 17:49:56,2017-08-01 18:17:47
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-01 21:13:01,2017-08-09 18:18:43


#### Checking if packages were delivered with a delay

In [20]:
delivered_orders = df_orders[
    df_orders['order_delivered_customer_date'].notna()
]

late_orders = df_orders[
    df_orders['order_delivered_customer_date'].dt.normalize() >
    df_orders['order_estimated_delivery_date'].dt.normalize()   
]

late_percentage = (
    len(late_orders) / len(delivered_orders) * 100
)

print(f'Delivered orders: {len(delivered_orders):,}')
print(f'Late orders: {len(late_orders):,}')
print(f'Late delivery rate: {late_percentage:.2f}%')

Delivered orders: 96,476
Late orders: 6,535
Late delivery rate: 6.77%


In [21]:
late_orders[
    ['order_id', 'order_estimated_delivery_date',
     'order_delivered_customer_date']
].head(10)

,order_id,order_estimated_delivery_date,order_delivered_customer_date
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-09-28,2017-10-09 22:23:46
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-03-12,2018-03-21 22:03:54
41,6ea2f835b4556291ffdc53fa0b3b95e8,2017-12-21,2017-12-28 18:59:23
57,66e4624ae69e7dc89bd50222b59f581f,2018-04-02,2018-04-03 13:28:46
58,a685d016c8a26f71a0bb67821070e398,2017-03-30,2017-04-06 13:37:16
97,6a0a8bfbbe700284feb0845d95e0867f,2017-12-11,2017-12-28 19:43:00
102,a5474c0071dd5d1074e12d417078bbd0,2018-08-02,2018-08-03 19:28:47
110,9d531c565e28c3e0d756192f84d8731f,2017-12-22,2018-01-23 21:38:52
115,8fc207e94fa91a7649c5a5dab690272a,2017-12-19,2018-01-20 13:42:22
143,33a3edb84b9df4cb49546859b990ac6d,2018-03-16,2018-03-22 00:03:53


# 3. Products Dataset

In [22]:
df_products = pd.read_csv(
    csv_dict["olist_products_dataset"]
)

print("Shape:", df_products.shape)
display(df_products.head())



Shape: (32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [23]:
display(column_summary(df_products))

print(
    "Fully duplicated rows:",
    df_products.duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
product_id,str,0,0,32951
product_category_name,str,610,32877,73
product_name_lenght,float64,610,32884,66
product_description_lenght,float64,610,29990,2960
product_photos_qty,float64,610,32931,19
product_weight_g,float64,2,30746,2204
product_length_cm,float64,2,32851,99
product_height_cm,float64,2,32848,102
product_width_cm,float64,2,32855,95


Fully duplicated rows: 0


In [24]:
df_products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [25]:
missing_products = df_products[
    df_products['product_category_name'].isna()
]

display(missing_products)

missing_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

display(
    df_products[missing_columns]
    .isna()
    .value_counts()
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


product_category_name  product_name_lenght  product_description_lenght  product_photos_qty
False                  False                False                       False                 32341
True                   True                 True                        True                    610
Name: count, dtype: int64

In [26]:
print(
    'Duplicated product IDs:',
    df_products['product_id'].duplicated().sum()
)

Duplicated product IDs: 0


In [27]:
numeric_columns = df_products.select_dtypes(
    include="number"
).columns

print("Negative values per numeric column:")

display(
    (df_products[numeric_columns] < 0).sum()
)

Negative values per numeric column:


product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

# 4. Sellers Dataset

In [28]:
df_sellers = pd.read_csv(
    csv_dict["olist_sellers_dataset"]
)

print("Shape:", df_sellers.shape)
display(df_sellers.head())



Shape: (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [29]:
display(column_summary(df_sellers))

print(
    "Fully duplicated rows:",
    df_sellers.duplicated().sum()
)

print(
    "Duplicated seller IDs:",
    df_sellers["seller_id"].duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
seller_id,str,0,0,3095
seller_zip_code_prefix,int64,0,849,2246
seller_city,str,0,2484,611
seller_state,str,0,3072,23


Fully duplicated rows: 0
Duplicated seller IDs: 0


In [30]:
df_sellers["seller_state"].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [31]:
seller_locations = (
    df_sellers[["seller_city", "seller_state"]]
    .value_counts()
    .reset_index())

seller_locations.index += 1

display(seller_locations)

,seller_city,seller_state,count
1,sao paulo,SP,694
2,curitiba,PR,124
3,rio de janeiro,RJ,93
4,belo horizonte,MG,66
5,ribeirao preto,SP,52
...,...,...,...
632,aparecida de goiania,GO,1
633,bandeirantes,PR,1
634,vitoria de santo antao,PE,1
635,palotina,PR,1


# 5. Order Items Dataset

In [32]:
df_order_items = pd.read_csv(
    csv_dict["olist_order_items_dataset"]
)

print("Shape:", df_order_items.shape)

display(df_order_items.head())

Shape: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [33]:
display(column_summary(df_order_items))

,data_type,missing_values,repeated_values,unique_values
order_id,str,0,13984,98666
order_item_id,int64,0,112629,21
product_id,str,0,79699,32951
seller_id,str,0,109555,3095
shipping_limit_date,str,0,19332,93318
price,float64,0,106682,5968
freight_value,float64,0,105651,6999


### Investigating price and freight value

#### 1.Descriptive Statistics

In [34]:
columns_order_items = ['price', 'freight_value']

display(df_order_items[columns_order_items].describe())

display(check_numeric_values(
    df_order_items, 
    ['price', 'freight_value']
))

display(check_outliers(df_order_items, columns_order_items))

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


,missing,negative,zero,min,maximum,mean,median
price,0,0,0,0.85,6735.00,120.65,74.99
freight_value,0,0,383,0.00,409.68,19.99,16.26


,lower_bound,upper_bound,outlier_count
price,-102.600,277.400,8427
freight_value,0.975,33.255,12134


In [35]:
for col in columns_order_items:
    print(f"Top 10 records by {col}:")
    display(df_order_items.nlargest(10, col))

Top 10 records by price:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,2017-02-16 20:37:36,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,2018-08-02 04:05:13,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,2017-06-15 02:45:17,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,2017-04-18 13:25:18,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,2017-05-09 15:50:15,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,1,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,2018-06-28 12:36:36,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,2018-08-16 14:24:28,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,1,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,2018-04-05 08:27:27,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,1,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,2018-08-02 08:15:14,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,1,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,2017-03-23 20:08:04,3999.90,17.01


Top 10 records by freight_value:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
73486,a77e1550db865202c56b19ddc6dc4d53,1,ec31d2a17b299511e7c8627be9337b9b,257e61d3251fb5efb9daadddbc2cf7ca,2018-04-25 06:55:13,979.00,409.68
3303,076d1555fb53a89b0ef4d529e527a0f6,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2018-08-07 04:35:22,2338.08,375.28
28044,3fde74c28a3d5d618c00f26d51baafa0,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2018-08-22 03:15:09,2338.08,375.28
69797,9f49bd16053df810384e793386312674,1,256a9c364b75753b97bee410c9491ad8,5c030029b5916fed0986310385ec9009,2018-08-02 03:44:28,1149.00,339.59
16731,264a7e199467906c0727394df82d1a6a,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,2018-04-23 17:15:10,1050.00,338.30
87936,c7a07ddd52bbe18b61da49a8d89853d3,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,2018-07-31 10:45:11,1050.00,322.10
5037,0b6230647ed16f4b3e70282dc4b5b87f,1,46e24ce614899e36617e37ea1e4aa6ff,17f51e7198701186712e53a39c564617,2018-06-01 19:14:24,1050.00,321.88
3584,0822bcde10bb5d023755a71bc8f7797f,1,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,2018-05-10 23:52:39,990.00,321.46
29787,43bdbd9dc0931d72befdf4765af6c442,1,7e53e051875b2a0c9f22acd8a9a29a20,eeb6de78f79159600292e314a77cbd18,2018-07-12 08:35:21,3089.00,317.47
48320,6ddfbf514959b49b6410c01ad93054bb,1,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,2017-11-30 15:10:57,1045.00,314.40


#### 2.Checking if any shipping cost is higher then the item itself

In [36]:
suspicious_freight = df_order_items[
    df_order_items['freight_value'] > df_order_items['price']
]

display(suspicious_freight)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
58,0025081dcf9330f9a5052ae82c6ce396,1,4e3f399366b0047a572b6682f9bb166e,5f3ae9136c875522250f8184f253413a,2018-04-02 16:35:20,14.95,18.23
80,002f98c0f7efd42638ed6100ca699b42,1,d41dc2f2979f52d75d78714b378d4068,7299e27ed73d2ad986de7f7c77d919fa,2017-08-10 09:30:15,8.99,32.57
110,003edccf16bc5ec447f592913b3df2b4,1,500870614ddcf5bd84f7d26861026c8a,ef506c96320abeedfb894c34db06f478,2018-07-12 10:15:21,14.00,50.85
125,00482f2670787292280e0a8153d82467,1,a9c404971d1a5b1cbc2e4070e02731fd,702835e4b785b67a084280efca355756,2017-02-17 16:18:07,7.60,10.96
156,00602f25bffa1dcfb71e202fbf9824fb,1,32a8448d1612773bcfd0c5a8dd235e4e,86ccac0b835037332a596a33b6949ee1,2017-11-08 02:31:17,39.90,54.02
...,...,...,...,...,...,...,...
112575,ffd6f465cab015b452d398ac1f5d978a,2,fdcf45aa23bb8312ecc0027d6e1ef1c4,113e3a788b935f48aad63e1c41dac1bd,2018-05-03 09:30:37,7.84,22.06
112576,ffd6f465cab015b452d398ac1f5d978a,3,fdcf45aa23bb8312ecc0027d6e1ef1c4,113e3a788b935f48aad63e1c41dac1bd,2018-05-03 09:30:37,7.84,22.06
112626,fff2cdc825f9fc0ba3c04227cfa02303,1,044f05bc9de36e8a693a83e4bc79dd0d,ea8482cd71df3c1969d7b9473ff13abc,2018-02-08 02:52:42,24.99,25.63
112638,fffb0b1a50e65c449020434fa835e078,1,e7be84ea9462aac5e2b0b08eb35ba7f1,36a968b544695394e4e9d7572688598f,2017-04-28 16:45:12,4.90,10.96


In [37]:
print('Duplicated (order_id, order_item_id) pairs:',
df_order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum())

Duplicated (order_id, order_item_id) pairs: 0


In [38]:
original_dates = df_order_items['shipping_limit_date']

parsed_dates = pd.to_datetime(
    original_dates,
    errors='coerce'
)

invalid_dates = (
    parsed_dates.isna() & original_dates.notna()
).sum()

print('Invalid dates:', invalid_dates)

Invalid dates: 0


# 6. Order payment

In [39]:
df_order_payments = pd.read_csv(
    csv_dict["olist_order_payments_dataset"]
)

print("Shape:", df_order_payments.shape)
display(df_order_payments.head())

Shape: (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [40]:
display(column_summary(df_order_payments))

print(
    "Fully duplicated rows:",
    df_order_payments.duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
order_id,str,0,4446,99440
payment_sequential,int64,0,103857,29
payment_type,str,0,103881,5
payment_installments,int64,0,103862,24
payment_value,float64,0,74809,29077


Fully duplicated rows: 0


In [41]:
df_order_payments['payment_type'].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [42]:
numeric_columns = [
    "payment_value",
    "payment_installments",
    "payment_sequential"
]

display(df_order_payments[numeric_columns].describe())

display(check_numeric_values(
    df_order_payments,
    numeric_columns
))

,payment_value,payment_installments,payment_sequential
count,103886.000000,103886.000000,103886.000000
mean,154.100380,2.853349,1.092679
std,217.494064,2.687051,0.706584
min,0.000000,0.000000,1.000000
25%,56.790000,1.000000,1.000000
50%,100.000000,1.000000,1.000000
75%,171.837500,4.000000,1.000000
max,13664.080000,24.000000,29.000000


,missing,negative,zero,min,maximum,mean,median
payment_value,0,0,9,0.0,13664.08,154.10,100.0
payment_installments,0,0,2,0.0,24.00,2.85,1.0
payment_sequential,0,0,0,1.0,29.00,1.09,1.0


In [43]:
order_ids = df_order_payments.loc[
    df_order_payments["payment_sequential"] == 29,
    "order_id"
]

display(
    df_order_payments[
        df_order_payments["order_id"].isin(order_ids)
    ].sort_values(["order_id", "payment_sequential"])
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


In [44]:
df_order_payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

np.int64(0)

In [45]:
# checking payment sequence continuity

sequence_check = df_order_payments.groupby("order_id")["payment_sequential"].agg(
    count="count",
    min="min",
    max="max",
    unique="nunique"
)

invalid_sequences = sequence_check[
    (sequence_check["min"] != 1) |
    (sequence_check["max"] != sequence_check["count"]) |
    (sequence_check["unique"] != sequence_check["count"])
]

display(invalid_sequences)
display(
    invalid_sequences.value_counts(
    subset=["count", "min", "max", "unique"]
))

,count,min,max,unique
order_id,,,,
00ac05fe0fc047c54418098eb64e3aaa,1,2,2,1
056c68d093c100017aab1f00f260705c,1,2,2,1
0668d086b3ae41adde3aed3dacbc8fae,1,2,2,1
0a7d898c6305101e69e9f5a05cd0130d,1,2,2,1
0d0e88a418636d40ea780339701db49c,1,2,2,1
...,...,...,...,...
f0f94b7c7548150f33f5d9e7597d396f,1,2,2,1
f17a58e669a7f4cf7a7f227899f8abf7,1,2,2,1
f5691c2b1ca263490374d13d020bd950,1,2,2,1


count  min  max  unique
1      2    2    1         78
2      2    3    2          2
Name: count, dtype: int64

In [46]:
# checking for outliers

display(check_outliers(
    df_order_payments,
    numeric_columns
))

,lower_bound,upper_bound,outlier_count
payment_value,-115.78125,344.40875,7981
payment_installments,-3.50000,8.50000,6313
payment_sequential,1.00000,1.00000,4526


In [47]:
for col in numeric_columns:
    print(f"Top 10 records by {col}:")
    display(df_order_payments.nlargest(10, col))

Top 10 records by payment_value:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66
62409,2cc9089445046817a7539d90805e6e5a,1,boleto,1,6081.54
43232,a96610ab360d42a2e5335a3998b4718a,1,credit_card,10,4950.34
70320,b4c4b76c642808cbe472a32b86cddc95,1,credit_card,5,4809.44
6440,199af31afc78c699f0dbf71fb178d4d4,1,credit_card,8,4764.34
67546,8dbc85d1447242f3b127dda390d56e19,1,credit_card,8,4681.78


Top 10 records by payment_installments:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
2970,70b7e94ea46d3e8b5bc12a50186edaf0,1,credit_card,24,274.84
10791,859f516f2fc3f95772e63c5757ab0d5b,1,credit_card,24,609.56
12307,ff36cbc44b8f228e0449c92ef089c843,1,credit_card,24,756.49
18512,2b7dbe9be72b8f9733844c31055c0825,1,credit_card,24,345.39
21713,6ae2e8b8fac02522481d2a2f4ca4412c,1,credit_card,24,433.43
23024,90f864fe19d11549fa01eb81c4dd87e3,1,credit_card,24,588.58
36088,84d2098c97827c6327ed4d7be95e1fc8,1,credit_card,24,286.78
50401,ffb18bf111fa70edf316eb0390427986,1,credit_card,24,617.24
52846,63dbe0c8e63e5f1b4deec09d4f044a7f,1,credit_card,24,771.69
55094,fcbb6af360b31b05460c2c8e524588c0,1,credit_card,24,1194.38


Top 10 records by payment_sequential:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
39108,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.26
39111,fa65dad1b0e818e3ccc5cb0e39231352,28,voucher,1,29.05
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
32393,ccf804e764ed5650cd8759557269dc13,26,voucher,1,23.10
79587,fa65dad1b0e818e3ccc5cb0e39231352,26,voucher,1,28.27
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
39132,ccf804e764ed5650cd8759557269dc13,25,voucher,1,1.53
51816,ccf804e764ed5650cd8759557269dc13,24,voucher,1,2.79
99213,fa65dad1b0e818e3ccc5cb0e39231352,24,voucher,1,0.42
60241,ccf804e764ed5650cd8759557269dc13,23,voucher,1,1.03


# 7. Order Reviews 

In [48]:
df_order_reviews = pd.read_csv(
    csv_dict["olist_order_reviews_dataset"]
)

print("Shape:", df_order_reviews.shape)
display(df_order_reviews.head())



Shape: (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [49]:
display(column_summary(df_order_reviews))

print(
    "Fully duplicated rows:",
    df_order_reviews.duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
review_id,str,0,814,98410
order_id,str,0,551,98673
review_score,int64,0,99219,5
review_comment_title,str,87656,94696,4527
review_comment_message,str,58247,63064,36159
review_creation_date,str,0,98588,636
review_answer_timestamp,str,0,976,98248


Fully duplicated rows: 0


In [50]:
df_order_reviews['review_score'].value_counts(dropna=False).sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [51]:
# converting dates to timestamps 
review_date_columns = [
    'review_creation_date',
    'review_answer_timestamp'
]

df_order_reviews[review_date_columns] = (
    df_order_reviews[review_date_columns].apply(pd.to_datetime)
)

In [52]:
invalid_dates = df_order_reviews[
    df_order_reviews['review_answer_timestamp'] < 
    df_order_reviews['review_creation_date']
]

print(f'Invalid review date sequences: {len(invalid_dates)}')
display(invalid_dates)


Invalid review date sequences: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


# 8. Geo-location

In [53]:
df_geolocation = pd.read_csv(
    csv_dict["olist_geolocation_dataset"]
)

print("Shape:", df_geolocation.shape)
display(df_geolocation.head())

Shape: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [54]:
display(column_summary(df_geolocation))

,data_type,missing_values,repeated_values,unique_values
geolocation_zip_code_prefix,int64,0,981148,19015
geolocation_lat,float64,0,282791,717372
geolocation_lng,float64,0,282548,717615
geolocation_city,str,0,992152,8011
geolocation_state,str,0,1000136,27


In [55]:
prefix_lengths = (
    df_geolocation['geolocation_zip_code_prefix']
    .astype(str)
    .str.len()
)

print('ZIP code prefix lengths:')
display(prefix_lengths.value_counts().sort_index())

print('Prefix shorter than 5 digits:', (prefix_lengths < 5).sum())

display(df_geolocation[prefix_lengths < 5].head(10))

ZIP code prefix lengths:


geolocation_zip_code_prefix
4    245733
5    754430
Name: count, dtype: int64

Prefix shorter than 5 digits: 245733


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
5,1012,-23.547762,-46.635361,são paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP


In [56]:
geolocation_states = (
    df_geolocation["geolocation_state"]
    .value_counts()
    .reset_index()
)

geolocation_states.index = geolocation_states.index + 1

display(geolocation_states)

,geolocation_state,count
1,SP,404268
2,MG,126336
3,RJ,121169
4,RS,61851
5,PR,57859
6,SC,38328
7,BA,36045
8,GO,20139
9,ES,16748
10,PE,16432


In [57]:
# checking whether coordinates fall inside the expected values 

invalid_coordinates = df_geolocation[
    (~df_geolocation["geolocation_lat"].between(-90, 90)) |
    (~df_geolocation["geolocation_lng"].between(-180, 180))
]

print('Invalid coordinates:', len(invalid_coordinates))
display(invalid_coordinates)


Invalid coordinates: 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state


In [58]:
# checking whether the coordinates fall in the brazilian territory
outside_brazil = df_geolocation[
    (~df_geolocation["geolocation_lat"].between(-34, 6)) |
    (~df_geolocation["geolocation_lng"].between(-74, -34))
]

print('Coordinates outside Brazil\'s approximate bounds:', len(outside_brazil))
display(outside_brazil)

Coordinates outside Brazil's approximate bounds: 42


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


# 9. Product Category Name Translation

In [59]:
df_translation = pd.read_csv(
    csv_dict["product_category_name_translation"]
)

print("Shape:", df_translation.shape)
display(df_translation.head())

Shape: (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [60]:
display(column_summary(df_translation))

print(
    "Fully duplicated rows:",
    df_translation.duplicated().sum()
)

,data_type,missing_values,repeated_values,unique_values
product_category_name,str,0,0,71
product_category_name_english,str,0,0,71


Fully duplicated rows: 0


In [61]:
missing_translations = set(
    df_products["product_category_name"].dropna()
) - set(
    df_translation["product_category_name"]
)

print(missing_translations)
print(f"Categories without translations: {len(missing_translations)}")

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}
Categories without translations: 2
